# Two-stage QPUF — `m2 − m1` consistency (debias vs no-mitig)

The **two-stage QPUF** runs two QPE stages on the same Haar-random `U` and compares their precision registers per shot:

- `m1` = stage-1 register (prec_a)
- `m2` = stage-2 register (prec_b)

The QPUF response is the **per-shot difference `m2 − m1`**. Ideally `m1 = m2` (both stages read the same collapsed eigenphase), so `m2 − m1` clusters at **0**. Noise spreads it; **debiasing should tighten it back toward the ideal**.

> Even the noiseless ideal does **not** give a 100% match rate: for a Haar-random `U` the eigenphases don't land on the `n_prec`-bit QPE grid, so `m1` and `m2` are independent QPE samples with intrinsic spread. The right reference is therefore the **local-simulator** `m2 − m1` distribution, not a delta spike at 0.

Data comes from `job_results_2stage/` (written by `ionq_qpuf_two_stage.py`).

In [ ]:
import os
import sys
import glob
import json
from collections import defaultdict
from datetime import datetime as _dt

import numpy as np
import matplotlib.pyplot as plt
from qiskit.quantum_info import Statevector

# Force consistent black-on-white styling so titles/labels are always legible,
# regardless of the editor's light/dark theme (a dark theme would otherwise make
# the text white and invisible on a light background).
plt.rcParams.update({
    "figure.facecolor": "white", "savefig.facecolor": "white",
    "axes.facecolor": "white",
    "text.color": "black", "axes.titlecolor": "black", "axes.labelcolor": "black",
    "axes.edgecolor": "black", "xtick.color": "black", "ytick.color": "black",
})

# Make ionq_noise_mitig.py importable regardless of the kernel's working dir.
_MODDIR = next((d for d in (
    os.getcwd(),
    os.path.join(os.getcwd(), "noise_mitigation"),
    os.path.expanduser("~/python/QUEN_QPUF/noise_mitigation"),
) if os.path.isfile(os.path.join(d, "ionq_noise_mitig.py"))), None)
if _MODDIR is None:
    raise FileNotFoundError("Could not locate ionq_noise_mitig.py -- add its folder to sys.path manually.")
if _MODDIR not in sys.path:
    sys.path.insert(0, _MODDIR)

import ionq_noise_mitig as submit   # build_qpuf_two_stage, haar sampler, ...

RESULTS_DIR = os.path.join(_MODDIR, "job_results_2stage")
PLOTS_DIR = os.path.join(_MODDIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

# ============================ CHOOSE WHICH SET TO PLOT ========================
# Results are grouped by (n_prec, n_targ). Pick a family here, or leave both
# None to auto-select the family of your MOST RECENTLY submitted job. To force
# exact files instead, set MITIG_FILE / NOMITIG_FILE to filenames in RESULTS_DIR.
SELECT_N_PREC = None     # e.g. 4 or 2 ; None = auto (latest submission's family)
SELECT_N_TARG = None     # e.g. 1 or 2 ; None = infer from SELECT_N_PREC
MITIG_FILE = None        # e.g. "9ddd87d6-...json"  (debias)
NOMITIG_FILE = None      # e.g. "908c0d58-...json"  (no-mitig)
# =============================================================================


def _mitig(res):
    em = res.get("error_mitigation")
    if em is not None:
        return em
    return "debias" if "debias" in res.get("circuit_type", "") else "none"


def _sortkey(res):
    try:
        return (_dt.fromisoformat(res["datetime"]), res["_file"])
    except Exception:
        return (_dt.min, res["_file"])


all_res = []
for path in sorted(glob.glob(os.path.join(RESULTS_DIR, "*.json"))):
    r = json.load(open(path)); r["_file"] = os.path.basename(path); r["_mitig"] = _mitig(r)
    all_res.append(r)
if not all_res:
    raise FileNotFoundError(f"No result JSONs in {RESULTS_DIR}/ (run ionq_qpuf_two_stage.py).")

families = defaultdict(list)
for r in all_res:
    families[(r["n_prec"], r["n_targ"])].append(r)

print("Available two-stage families in job_results_2stage/:")
for k in sorted(families):
    g = families[k]
    print(f"  n_prec={k[0]:>2}, n_targ={k[1]}:  "
          f"debias={sum(x['_mitig']=='debias' for x in g)}  nomitig={sum(x['_mitig']=='none' for x in g)}")

by_file = {r["_file"]: r for r in all_res}
if MITIG_FILE or NOMITIG_FILE:
    debias_res = by_file.get(MITIG_FILE)
    nomitig_res = by_file.get(NOMITIG_FILE)
else:
    if SELECT_N_PREC is not None:
        cands = [k for k in families if k[0] == SELECT_N_PREC
                 and (SELECT_N_TARG is None or k[1] == SELECT_N_TARG)]
        if not cands:
            raise ValueError(f"No family with n_prec={SELECT_N_PREC}"
                             + (f", n_targ={SELECT_N_TARG}" if SELECT_N_TARG is not None else "")
                             + f". Available: {sorted(families)}")
        fam = max(cands, key=lambda k: k[1])     # if >1 n_targ, take the larger
    else:
        latest = max(all_res, key=_sortkey)       # family of the newest submission
        fam = (latest["n_prec"], latest["n_targ"])
    grp = families[fam]
    deb = [r for r in grp if r["_mitig"] == "debias"]
    nom = [r for r in grp if r["_mitig"] == "none"]
    debias_res = max(deb, key=_sortkey) if deb else None
    nomitig_res = max(nom, key=_sortkey) if nom else None

if debias_res is None or nomitig_res is None:
    raise FileNotFoundError("Need BOTH a debias and a no-mitig result for the selected family "
                            "(run ionq_qpuf_two_stage.py with USE_DEBIAS True and False).")

# never compare mismatched circuits
for fld in ("n_prec", "n_targ", "seed", "target_init_seed"):
    if debias_res.get(fld) != nomitig_res.get(fld):
        raise ValueError(f"debias and no-mitig differ in {fld!r} -- not the same circuit. "
                         "Set SELECT_N_PREC / MITIG_FILE / NOMITIG_FILE explicitly.")

ref = debias_res
n_prec, n_targ = ref["n_prec"], ref["n_targ"]
n_qubits = n_targ + 2 * n_prec
M = 2 ** n_prec   # m1, m2 each range 0 .. M-1
print(f"\nSelected: N_PREC={n_prec}, N_TARG={n_targ}  ({n_qubits} qubits), "
      f"{ref['n_shots']} shots, est. F={ref.get('est_fidelity', float('nan')):.2f}")
print(f"  no-mitig: {nomitig_res['_file']}")
print(f"  debias  : {debias_res['_file']}")

## Extract (m1, m2) per shot + the local-simulator reference

Each measured bitstring is `c[2·n_prec−1]…c[0]`: the **right** `n_prec` bits are `prec_a` (= `m1`), the **left** `n_prec` bits are `prec_b` (= `m2`). We decode both to integers and tally `(m1, m2)`. The **match rate** `P(m1 == m2)` is the headline QPUF reproducibility metric.

In [ ]:
def split_m(bitstring, n):
    """Hardware bitstring c[2n-1]..c[0] -> (m1, m2). prec_a=c[0..n-1]=m1 (right
    half), prec_b=c[n..2n-1]=m2 (left half); reverse each half so prec[0]=MSB."""
    m2 = int(bitstring[:n][::-1], 2)
    m1 = int(bitstring[n:][::-1], 2)
    return m1, m2


def hw_joint(res, n):
    """{(m1, m2): probability} from a hardware counts dict."""
    tot = sum(res["counts"].values())
    j = defaultdict(float)
    for s, c in res["counts"].items():
        j[split_m(s, n)] += c / tot
    return dict(j)


def ideal_joint(res, n, n_targ):
    """{(m1, m2): probability} from the noiseless two-stage statevector."""
    U = np.array(res["unitary"]["real"]) + 1j * np.array(res["unitary"]["imag"])
    qc = submit.build_qpuf_two_stage(n, n_targ, U, res["target_init_seed"])
    qc = qc.remove_final_measurements(inplace=False)
    sv = Statevector(qc)
    N = n_targ + 2 * n
    # qubit layout: q[0..n_targ-1]=targ, q[n_targ..]=prec_a, q[n_targ+n..]=prec_b
    j = defaultdict(float)
    for key, p in sv.probabilities_dict().items():       # key = q[N-1]..q[0]
        a = "".join(key[N - 1 - (n_targ + k)] for k in range(n))         # prec_a, MSB first
        b = "".join(key[N - 1 - (n_targ + n + k)] for k in range(n))     # prec_b
        j[(int(a, 2), int(b, 2))] += p
    return dict(j)


def match_rate(joint):
    return sum(p for (a, b), p in joint.items() if a == b)


def mean_abs_delta(joint):
    return sum(abs(b - a) * p for (a, b), p in joint.items())


JOINTS = {
    "Local sim": ideal_joint(ref, n_prec, n_targ),
    "HW no-mitig": hw_joint(nomitig_res, n_prec),
    "HW debias": hw_joint(debias_res, n_prec),
}

print(f"{'run':12s} {'match P(m1=m2)':>15} {'mean|m2-m1|':>12}")
print("-" * 41)
for name, j in JOINTS.items():
    print(f"{name:12s} {match_rate(j):>15.3f} {mean_abs_delta(j):>12.2f}")

mr = {k: match_rate(v) for k, v in JOINTS.items()}
gap = mr["Local sim"] - mr["HW no-mitig"]
recovered = (mr["HW debias"] - mr["HW no-mitig"]) / gap if gap else float("nan")
print(f"\nDebias recovers {recovered*100:.0f}% of the no-mitig->ideal match-rate gap "
      f"({mr['HW no-mitig']:.3f} -> {mr['HW debias']:.3f}, ideal {mr['Local sim']:.3f})")

## 3-D view: `m1` vs `m2 − m1` vs frequency

One 3-D spike chart per run, in the style of the reference figure: **X = `m1`** (stage-1 readout), **Y = `m2 − m1`**, **Z = frequency of outcome (%)**. Spikes are **coloured by `m1`** (yellow → blue). A consistent QPUF concentrates its weight on the `m2 − m1 = 0` plane. Followed by the acceptance-vs-`delta` curve.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

CIRCUIT_INFO = (f"two-stage QPUF  |  N_PREC={n_prec}, N_TARG={n_targ} ({n_qubits} qubits)"
                f"  |  {ref['n_shots']} shots"
                + (f"  |  est. F={ref['est_fidelity']:.2f}" if ref.get('est_fidelity') else "")
                + f"  |  {ref.get('qpu', '?')}")
FNAME_SUFFIX = f"2stage_nprec{n_prec}_ntarg{n_targ}_{ref['n_shots']}shots"

order = ["Local sim", "HW no-mitig", "HW debias"]
CMAP = plt.get_cmap("plasma_r")     # low m1 -> yellow, high m1 -> blue (reference style)


def plot_joint_3d(joint, title, ax):
    """3D spikes coloured by m1: X=m1, Y=m2-m1, Z=frequency(%)."""
    agg = defaultdict(float)
    for (m1, m2), p in joint.items():
        agg[(m1, m2 - m1)] += p
    xs = np.array([k[0] for k in agg], dtype=float)
    ys = np.array([k[1] for k in agg], dtype=float)
    zs = np.array([agg[k] * 100.0 for k in agg])           # frequency (%)
    colors = CMAP(xs / max(M - 1, 1))
    w = 0.30
    ax.bar3d(xs - w / 2, ys - w / 2, np.zeros_like(zs), w, w, zs,
             color=colors, shade=True, zsort="average")
    ax.set_xlabel(r"$m_1$  (stage 1)")
    ax.set_ylabel(r"$m_2 - m_1$")
    ax.set_zlabel("Frequency of outcome (%)")
    ax.set_title(title, fontsize=11)
    ax.view_init(elev=30, azim=-50)


fig = plt.figure(figsize=(20, 6.5))
for i, name in enumerate(order):
    ax = fig.add_subplot(1, 3, i + 1, projection="3d")
    plot_joint_3d(JOINTS[name], f"{name}\nmatch P(m1=m2) = {match_rate(JOINTS[name]):.3f}", ax)
fig.suptitle(f"Two-stage QPUF: m1 vs (m2 - m1) vs frequency\n{CIRCUIT_INFO}", y=1.0, fontsize=13)
out3d = os.path.join(PLOTS_DIR, f"qpuf_{FNAME_SUFFIX}_3d.png")
fig.savefig(out3d, dpi=150, bbox_inches="tight")
print("saved:", out3d)
plt.show()

In [ ]:
# braket-style acceptance vs delta: P(cyclic |m1 - m2| <= delta) for each run.
def cyclic_distance(a, b, n_prec):
    """Wrap-around distance between two QPE integers (the m-axis is periodic)."""
    d = abs(a - b)
    return min(d, 2 ** n_prec - d)


series_colors = {"Local sim": "#1f77b4", "HW no-mitig": "#d62728", "HW debias": "#2ca02c"}
ds = list(range(0, M + 1))

fig, ax = plt.subplots(figsize=(8.5, 4.6))
for name in order:
    acc = [sum(p for (m1, m2), p in JOINTS[name].items()
               if cyclic_distance(m1, m2, n_prec) <= d) for d in ds]
    ax.plot(ds, acc, marker="o", ms=3, color=series_colors[name],
            label=f"{name}  (match {match_rate(JOINTS[name]):.3f})")
ax.axvline(2, color="gray", ls="--", lw=1, label="delta=2")
ax.set_xlabel("delta  (cyclic |m1 - m2| threshold)")
ax.set_ylabel("acceptance rate  P(|m1-m2|_cyc <= delta)")
ax.set_title(f"Two-stage QPUF acceptance vs delta\n{CIRCUIT_INFO}", fontsize=11)
ax.legend()
fig.tight_layout()
outacc = os.path.join(PLOTS_DIR, f"qpuf_{FNAME_SUFFIX}_acceptance.png")
fig.savefig(outacc, dpi=150, bbox_inches="tight")
print("saved:", outacc)
plt.show()